In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("subreddit_similarity_results.csv")

In [4]:
df

,Subreddit_A,Subreddit_B,Similarity_Score
0,196,19684,0.9989
1,AliexpressCouponFind,AliexpressCoupons2021,0.9986
2,AxieInfinityScholar,AxieScholarshipsPH,0.9971
3,BSCMoonShots,AllCryptoBets,0.9967
4,19684,197,0.9966
...,...,...,...
3225705,BeeSwarmSimulator,BostonRSS,0.3000
3225706,AppleFitnessPlus,4ChanMeta,0.3000
3225707,CosmicDisclosure,Azoozkie,0.3000
3225708,Baddies_of_NC,Bowyer,0.3000


In [5]:
df['pair'] = df.apply(lambda row: tuple(sorted([row['Subreddit_A'], row['Subreddit_B']])), axis=1)

df_unique = df.drop_duplicates(subset='pair')

In [6]:
threshold = df['Similarity_Score'].quantile(0.97)

In [7]:
top_pairs = df_unique[df_unique['Similarity_Score'] >= threshold]

In [9]:
top_pairs

,Subreddit_A,Subreddit_B,Similarity_Score,pair
0,196,19684,0.9989,"(196, 19684)"
1,AliexpressCouponFind,AliexpressCoupons2021,0.9986,"(AliexpressCouponFind, AliexpressCoupons2021)"
2,AxieInfinityScholar,AxieScholarshipsPH,0.9971,"(AxieInfinityScholar, AxieScholarshipsPH)"
3,BSCMoonShots,AllCryptoBets,0.9967,"(AllCryptoBets, BSCMoonShots)"
4,19684,197,0.9966,"(19684, 197)"
...,...,...,...,...
96795,CanberraNSFW,Cebu_r4r,0.7361,"(CanberraNSFW, Cebu_r4r)"
96796,CorpusChristiCasuals,Amazingfemale18plus,0.7361,"(Amazingfemale18plus, CorpusChristiCasuals)"
96797,AnimalRestaurant,AvatarMemes,0.7361,"(AnimalRestaurant, AvatarMemes)"
96798,BSCcryptoListings,AltStreetBets,0.7361,"(AltStreetBets, BSCcryptoListings)"


In [12]:
df["Subreddit_A"].unique()

<StringArray>
[                 '196', 'AliexpressCouponFind',  'AxieInfinityScholar',
         'BSCMoonShots',                '19684',            'CoinSales',
    'CouponCodeExplore',      'AbsoluteWeapons', 'BestPolishInfluencer',
            'CelebLBDs',
 ...
             'CVEWatch',           'BestOfEbay',           'ALLtheNFSW',
          'AfricaVoice',        'ControlTheory',    'ActualidadMundial',
   'AcademicPsychology',             'Angular2',           'BotBustLog',
                 'ACGN']
Length: 3608, dtype: str

In [9]:
from pyvis.network import Network
import networkx as nx

In [12]:
top_nodes = top_pairs.head(100)

In [18]:
G = nx.from_pandas_edgelist(
    top_nodes,
    source='Subreddit_A',
    target='Subreddit_B',
    edge_attr='Similarity_Score'
)

for u, v, d in G.edges(data=True):
    d['value'] = d['Similarity_Score']
    if d['Similarity_Score'] > 0.99:
        d['color'] = 'red'

degree_dict = dict(G.degree())

for node in G.nodes():
    G.nodes[node]['size'] = max(degree_dict[node] * 3, 5)

net = Network(
    height="800px",
    width="100%",
    bgcolor="#111111",
    font_color="white",
    notebook=False,
    cdn_resources="in_line"
)

net.from_nx(G)
net.force_atlas_2based()

html = net.generate_html()

with open("subreddit_graph.html", "w", encoding="utf-8") as f:
    f.write(html)